# Regresión con zarigüeyas

**Preámbulo**

---

## Instalar `pandas` y `seaborn` en Windows

1. Ejecute **Anaconda PowerShell**.
2. Active el entorno virtual `ml` con este comando: `conda activate ml`
3. Instale las librerías con este comando: `pip install pandas seaborn`
4. Verifique la instalación con este comando: `python -c "import pandas, seaborn; print('Instalación correcta')"`

---

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

## 1. Lectura de la base de datos

In [ ]:
datos = pd.read_csv('./possum.csv')

Para verificar la lectura de datos

In [ ]:
datos.head()

## 2. Curación de datos o curaduría de datos (data curation)

### 2.1 Eliminación de información no necesaria

In [ ]:
datos = datos.drop('case', axis=1)

In [ ]:
datos.head() # verificamos que case ya no existe

Información de los tipos de datos y su estructura

In [ ]:
datos.info()

Revisar cuántos valores distintos tiene cada variable

In [ ]:
datos.nunique()

### 2.2 Conversión de variables tipo categórica a numérica

Este comando convierte la columna `sex` en una variable numérica, asignando el valor `1` a los registros cuyo sexo sea `"f"` y `0` a los demás; en términos generales, esta transformación permite trabajar más fácilmente con la variable en análisis estadísticos, resúmenes por grupos o modelos, teniendo presente que en este caso el valor `1` representa femenino y `0` el otro grupo.

In [ ]:
datos["sex"] = [1 if i == "f" else 0 for i in datos["sex"]]

In [ ]:
datos.head() # verificamos sex

Transforme la variable Pop de categórica a numérica, reemplazando cada categoría por un código numérico

In [ ]:
# aquí su código

datos.head() # verificamos Pop

### 2.3 Conteo de valores nulos por variable

Este comando revisa todo el conjunto de datos `datos` para identificar cuántos valores faltantes hay en cada columna; primero, `isnull()` detecta cuáles posiciones están vacías o tienen valores nulos, y luego `sum()` cuenta cuántos de esos casos aparecen en cada variable, por lo que el resultado sirve para reconocer rápidamente qué columnas tienen datos incompletos y cuántos faltantes presenta cada una.

In [ ]:
datos.isnull().sum()

#### 2.3.1 Manejo de registros con valores nulos

Este comando selecciona únicamente las filas del DataFrame `datos` en las que la variable `footlgth` tiene un valor nulo o faltante; para ello, `datos["footlgth"].isnull()` identifica cuáles registros no tienen dato en esa columna, y luego `datos[...]` usa esa condición para filtrar y mostrar solo esos casos, lo que resulta útil para localizar, revisar y analizar los registros incompletos antes de limpiar o transformar la base de datos.

In [ ]:
datos[datos["footlgth"].isnull()]

Este comando elimina la fila cuyo índice es `40` del DataFrame `datos`; el parámetro `axis=0` indica que la operación se realiza sobre las filas, y `inplace=True` significa que el cambio se aplica directamente sobre el DataFrame original, sin necesidad de guardarlo en una nueva variable. En otras palabras, después de ejecutar esta instrucción, esa fila deja de existir en `data`.

In [ ]:
datos.drop([40], axis = 0, inplace=True)

Solucione las inconcistencias con `age`

In [ ]:
# aquí su código

Verificamos que los cambios son correctos

In [ ]:
datos.info()

In [ ]:
datos.isnull().sum()

## 3. Modelo de regresión

### 3.1 Selección de la característica(s) a modelar

In [ ]:
input_features = ["hdlngth"]  # <-- Modifique
output_feature = ["sex"]      # <-- no modificar

In [ ]:
X = datos[input_features]
X

In [ ]:
y = datos[output_feature].squeeze()    # convierte a Series
y

### 3.2 División de los datos (entrenamiento/validación)

In [ ]:
# Separación de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("X_train:", X_train.shape)       # tamaño por dimensión (filas, columnas)
print("y_train:", y_train.shape)       # tamaño por dimensión (filas, columnas)

print("X_test:", X_test.shape)       # tamaño por dimensión (filas, columnas)
print("y_test:", y_test.shape)       # tamaño por dimensión (filas, columnas)

### 3.3 Modelo de regresión

In [ ]:
# Selección del orden del polinomio
orden = 1 # <-- Modifique

# Pipeline: características polinómicas + regresión lineal
modelo = make_pipeline(
    PolynomialFeatures(degree=orden, include_bias=False),
    LinearRegression()
)

# Entrenamiento
modelo.fit(X_train, y_train)

### 3.4 Parámetros del modelo

In [ ]:
# Parámetros del modelo: v(m) = b0 + b1*m + b2*m^2 + ... + b_orden*m^orden
lin = modelo.named_steps["linearregression"]

b0 = lin.intercept_
b = lin.coef_  # [b1, b2, ..., b_orden] porque include_bias=False

# imprime coeficientes
print(f"Orden: {orden}")
print(f"b0: {b0:.6f}")
for i, bi in enumerate(b, start=1):
    print(f"b{i}: {bi:.6e}")

### 3. Evaluación/validación del modelo

In [ ]:
y_hat_train = modelo.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_hat_train))
r2_train = r2_score(y_train, y_hat_train)

# RMSE: error cuadrático medio (en voltios).
# R^2: coeficiente de determinación (0 a 1).
print(f"RMSE (entrenamiento): {rmse_train:.6f}")
print(f"R^2  (entrenamiento): {r2_train:.6f}")

In [ ]:
y_hat_test = modelo.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_hat_test))
r2_test = r2_score(y_test, y_hat_test)

# RMSE: error cuadrático medio (en voltios).
# R^2: coeficiente de determinación (0 a 1).
print(f"RMSE (validación): {rmse_test:.6f}")
print(f"R^2  (validación): {r2_test:.6f}")

---

## Actividad

A partir de los descriptores disponibles, determine el polinomio que mejor predice el sexo de una zarigüeya.- - 

- Explique el procedimiento utilizado y justifique la elección del modelo obtenido.